# $$\text{Ejercicio 1}$$

# $$\text{Dropout y Batch Normalization}$$

# 1. Importación librerías

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Aleatoriedad
import itertools
import random

# Preprocesamiento
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Deep Learning (Keras)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

# Configuración
plt.style.use('ggplot')
pd.set_option('display.max_columns', None)

In [14]:
import openpyxl

# 2. Carga y limpieza de datos

In [2]:
# Carga y Preprocesamiento
try:
    df = pd.read_csv('../data/bank-additional-full.csv', sep=';')
except FileNotFoundError:
    print("⚠️ Cargando dataset sintético para demostración...")
    from sklearn.datasets import make_classification
    X_dummy, y_dummy = make_classification(n_samples=5000, n_features=20, random_state=42)
    df = pd.DataFrame(X_dummy, columns=[f'feat_{i}' for i in range(20)])
    df['y'] = np.where(y_dummy==1, 'yes', 'no')

df['target'] = df['y'].apply(lambda x: 1 if x == 'yes' else 0)
df = df.drop(columns=['y'])

categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numerical_cols.remove('target')

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int) #no puede haber booleanos, ni strings

X = df_encoded.drop(columns=['target'])
y = df_encoded['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

print(f"Dataset listo: {X_train.shape[0]} muestras de entrenamiento, {X_train.shape[1]} características.")

Dataset listo: 32950 muestras de entrenamiento, 53 características.


# 3. Diseño de funciones con arquitectura de las redes

In [4]:
def build_modelos_prueba(input_dim, neuronas_capa1=256, neuronas_capa2=128, neuronas_capa3=64,
                        dropout_capa1=0.4, dropout_capa2=0.3, dropout_capa3=0.2,
                        f_activation1='relu', f_activation2='relu', f_activation3='relu'):
    # Arquitectura moderna: Dense -> BatchNorm -> Activation -> Dropout
    model = keras.Sequential([
        # Capa 1
        layers.Dense(neuronas_capa1, input_shape=(input_dim,)),
        layers.BatchNormalization(), # Estabiliza la distribución
        layers.Activation(f_activation1),
        layers.Dropout(dropout_capa1),
        
        # Capa 2
        layers.Dense(neuronas_capa2),
        layers.BatchNormalization(),
        layers.Activation(f_activation2),
        layers.Dropout(dropout_capa2),
        
        # Capa 3
        layers.Dense(neuronas_capa3),
        layers.BatchNormalization(),
        layers.Activation(f_activation3),
        layers.Dropout(dropout_capa3),
        
        # Salida
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])

    return model

# 4. Opciones de parámetros

In [5]:
INPUT_DIM = X_train.shape[1]
EPOCHS = 100 #Número de veces que el modelo verá todo el dataset (backpropagation - ir y venir)
BATCH_SIZE = 128 #Número de muestras que el modelo verá antes de actualizar los pesos (gradiente descendente)

In [6]:
neuronas_capa1=[256, 512]

neuronas_capa2=[128, 256]

neuronas_capa3=[64, 128]

dropout_capa1=[0.4, 0.8]

dropout_capa2=[0.3, 0.7]

dropout_capa3=[0.2, 0.6]

f_activation1=['relu', 'tanh']

f_activation2=['relu', 'tanh']

f_activation3=['relu', 'tanh']

In [7]:
# Crear todas las combinaciones posibles de parámetros
opciones_posibles = list(itertools.product(
    neuronas_capa1, neuronas_capa2, neuronas_capa3,
    dropout_capa1, dropout_capa2, dropout_capa3,
    f_activation1, f_activation2, f_activation3
))

In [8]:
# Seleccionar una muestra de combinaciones aleatorias
muestra_combinaciones = random.sample(opciones_posibles, 10)

# 5. Entrenamiento de modelos

In [ ]:
# print("2. Entrenando Modelo Robusto (con regularización)...")
# history_robust = model_robust.fit(
#     X_train, y_train,
#     validation_split=0.2,
#     epochs=EPOCHS,
#     batch_size=BATCH_SIZE,
#     verbose=0
# )
# print("¡Entrenamiento completado!")

In [10]:
resultados = []

for i, config in enumerate(muestra_combinaciones):
    n1, n2, n3, d1, d2, d3, a1, a2, a3 = config
    print(f"Probando configuración {i+1}/10: Neuronas({n1},{n2},{n3}) - Act({a1},{a2},{a3})")
    
    # Construir el modelo con la configuración actual
    model = build_modelos_prueba(
        input_dim=INPUT_DIM,
        neuronas_capa1=n1, neuronas_capa2=n2, neuronas_capa3=n3,
        dropout_capa1=d1, dropout_capa2=d2, dropout_capa3=d3,
        f_activation1=a1, f_activation2=a2, f_activation3=a3
    )
    
    # Entrenar (usamos pocas épocas para la búsqueda inicial)
    h = model.fit(
        X_train, y_train,
        validation_split=0.2,
        epochs=20, # Reducido para la prueba de múltiples modelos
        batch_size=BATCH_SIZE,
        verbose=0
    )
    
    # Extraer métricas finales
    res = {
        'n1': n1, 'n2': n2, 'n3': n3,
        'drop1': d1, 'drop2': d2, 'drop3': d3,
        'act1': a1, 'act2': a2, 'act3': a3,
        'final_auc': h.history['AUC'][-1],
        'final_val_auc': h.history['val_AUC'][-1],
        'final_loss': h.history['loss'][-1]
    }
    resultados.append(res)

# 3. Convertir a DataFrame
df_resultados = pd.DataFrame(resultados)

Probando configuración 1/10: Neuronas(512,256,64) - Act(relu,relu,relu)


c:\Users\HP ENVY\miniconda3\envs\mlnn2526\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Probando configuración 2/10: Neuronas(512,128,64) - Act(tanh,relu,relu)


c:\Users\HP ENVY\miniconda3\envs\mlnn2526\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Probando configuración 3/10: Neuronas(256,128,128) - Act(tanh,relu,relu)


...

c:\Users\HP ENVY\miniconda3\envs\mlnn2526\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Probando configuración 9/10: Neuronas(512,128,128) - Act(tanh,relu,relu)


c:\Users\HP ENVY\miniconda3\envs\mlnn2526\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Probando configuración 10/10: Neuronas(512,256,128) - Act(tanh,relu,relu)


c:\Users\HP ENVY\miniconda3\envs\mlnn2526\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
df_resultados.sort_values(by='final_val_auc', ascending=False)

,n1,n2,n3,drop1,drop2,drop3,act1,act2,act3,final_auc,final_val_auc,final_loss
7,256,256,128,0.4,0.3,0.2,relu,tanh,relu,0.948351,0.945907,0.171894
1,512,128,64,0.4,0.7,0.2,tanh,relu,relu,0.938449,0.944530,0.185434
6,512,256,128,0.4,0.3,0.6,tanh,tanh,tanh,0.940031,0.944516,0.184755
9,512,256,128,0.8,0.3,0.2,tanh,relu,relu,0.938323,0.942599,0.185702
2,256,128,128,0.4,0.7,0.2,tanh,relu,relu,0.937104,0.942475,0.187049
5,256,128,64,0.8,0.3,0.2,tanh,tanh,relu,0.931684,0.942169,0.193633
0,512,256,64,0.8,0.7,0.2,relu,relu,relu,0.934586,0.941671,0.190935
3,512,256,128,0.8,0.7,0.6,tanh,tanh,relu,0.929204,0.941586,0.196475
4,512,128,128,0.8,0.3,0.6,relu,relu,relu,0.935138,0.941576,0.190270
8,512,128,128,0.4,0.7,0.6,tanh,relu,relu,0.936039,0.940876,0.188823


In [15]:
# Guardar resultados en excel
df_resultados.to_excel('../results/grid_search_results.xlsx', index=False)